# Navier–Stokes: the learned operator on a test sample

We learn the map from a forcing $f$ to the vorticity $w(\cdot, T)$ of the 2-D incompressible Navier–Stokes equations on the torus (vorticity form, $w(0) = 0$, $\mathrm{Re} = 500$, $T = 5$), from a fixed dataset of 10000 training and 2000 test pairs on a $128 \times 128$ grid ([Zenodo 12825163](https://zenodo.org/records/12825163); see `raw_data/README.md`). We cannot choose where the inputs are, so the input measure is the empirical measure of the training forcings and we use `owls.EmpiricalWLS`, which samples optimally from the dataset.

* Inputs: PCA coefficients of the forcing that keep 99.5% of the variance (`encode_inputs`), scaled to $[-1, 1]$. Each coefficient gets a symmetric Jacobi measure with matching variance; it only selects the polynomial family (the basis is orthonormalized on the data).
* Outputs: Fourier coefficients in a basis orthonormal in $H^\alpha$, keeping 99.9% of the average $H^\alpha$ energy (`encode_outputs`).
* Approximation space: polynomial operators with an anisotropic hyperbolic cross index set, weights $w_j = \sigma_j / \sigma_1$ (normalized PCA singular values), univariate degree at most 10.

Here $\alpha = -2$ and the hyperbolic cross has $k = 28$. We show the true and predicted vorticity for the median-error test sample, and the distribution of relative $H^{-2}$ test errors.

**Status:** rerun after the cleanup, and the data are now saved in `data/`. The figure committed with arXiv v1 came from a run with $\alpha = -4$ instead of $-2$. See `ERRATA.md`.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
from ns_utils import load_ns_data, encode_inputs, encode_outputs
from ns_utils import HAlphaBasisCompressor, OperatorLearningPlotter

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
ALPHA = -2
HC_K = 28
MAX_DEGREE = 10
N_MAX = 50000
SEED = 0
if QUICK:
    HC_K = 4

In [ ]:
DATA_FILE = os.path.join(DATA, "learn_operator.npz")
if RECOMPUTE or QUICK:
    X_train, Y_train, X_test, Y_test = load_ns_data("raw_data")
    if QUICK:
        X_train, Y_train, X_test, Y_test = X_train[:600], Y_train[:600], X_test[:100], Y_test[:100]
    x_train, x_test, pca = encode_inputs(X_train, X_test, energy=0.995, random_state=SEED)
    measures = owls.alpha_from_variance(np.var(x_train, axis=0))   # symmetric Jacobi parameters matching each variance
    weights = pca.singular_values_ / pca.singular_values_[0]        # anisotropy weights
    print(f"d_in = {x_train.shape[1]} PCA modes")
    y_train, y_test, compressor = encode_outputs(Y_train, Y_test, alpha=ALPHA, energy=0.999)
    L = owls.index_set(x_train.shape[1], HC_K, "hc", weights, MAX_DEGREE, N_MAX)
    N = len(L)
    model = owls.EmpiricalWLS(x_train, y_train, L, measures).fit(int(N * np.log(N)), optimal=True, rng=SEED)
    test_err, rel_test_err = model.compute_errors(x_test, y_test)
    sample = np.argmin(np.abs(test_err - np.median(test_err)))
    np.savez(DATA_FILE, test_err=test_err, rel_test_err=rel_test_err,
             true_vorticity=compressor.reconstruct_spatial(y_test[sample]),
             approx_vorticity=compressor.reconstruct_spatial(model.predict(x_test[sample:sample + 1])[0]),
             rel_err_sample=rel_test_err[sample], err_sample=test_err[sample])
    print(f"d_in = {x_train.shape[1]}, d_out = {compressor.M}, N_eff = {N}, M = {model.M}, cond(G) = {model.cond():.2f}")
    print(f"Mean relative H^{ALPHA} test error: {np.mean(rel_test_err):.3e}")

In [ ]:
if not os.path.exists(DATA_FILE):
    print("No saved data: see figures/NS_med_err.png and figures/NS_test_err_dist.png for the arXiv v1 figures, or set RECOMPUTE = True.")
else:
    dat = np.load(DATA_FILE)
    plotter = OperatorLearningPlotter(domain=(0, 2 * np.pi, 0, 2 * np.pi), alpha=ALPHA)
    plotter.plot_comparison(dat["true_vorticity"], dat["approx_vorticity"], spectral_error=float(dat["err_sample"]),
                            rel_spectral_error=float(dat["rel_err_sample"]), title_prefix="Median Error Test Sample",
                            save_as=os.path.join(FIGS, "NS_med_err.png"))
    plotter.plot_error_distribution(dat["rel_test_err"], title_prefix="Relative Testing",
                                    save_as=os.path.join(FIGS, "NS_test_err_dist.png"))